Resumen: 
- Implementacion:
    - Training: fake y real hardware
    - Evaluation: noiseless con Aer
- Ejecucion via IBM Runtime EstimatorV2
- Batch parallelization mediante physical packing de copias logicas
- Qiskit transpiler elige el layout fisico y el routing

Embedding:
- Angle embedding
- Fake y real pubs sin EstimatorQNN ni TorchConnector.


## Implementation (real-hardware packed execution)


In [ ]:
#--- INSTALATION INSTRUCTIONS ---#

# For linux 64-bit systems,
#uname -a

# Conda quick installation
#mkdir -p ~/miniconda3
#wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh
#bash ~/miniconda3/miniconda.sh -b -u -p ~/miniconda3
#rm ~/miniconda3/miniconda.sh

# Create enviroment with conda
#conda create -n myenv python=3.10
#conda activate myenv
#pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-algorithms matplotlib pyyaml ipykernel
# IMPORTANT: Make sure you are on 3.10
# May need to restart the kernel after instalation

#--- Imports ---#
from qiskit import QuantumCircuit, qpy
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import efficient_su2, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import EstimatorV2 as EstimatorV2_rh, QiskitRuntimeService, Session
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as EstimatorV2_sim
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient, SPSAEstimatorGradient

import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import signal
import time
import yaml


from ipynb.fs.full.configuration_manager import 

In [ ]:
#- Create configuration file -#

# Set LOAD_CONFIG_FROM_FILE=True to ignore this dict and load configuration_file_path instead.
LOAD_CONFIG_FROM_FILE = False
configuration_file_path = "../data/test/qgan_rh_packed-q3-real-SPSA-seed0/config.yaml"

config_dict = {
    'implementation_options': {
        'n_qubits': 4,
        'seed': 0,
        'implementation': 'qgan_rh_ang',
        'execution_type': 'fake',  # 'fake' or 'real'
        'gradient_method': 'SPSA',  # 'SPSA' or 'PSR'
        'random_input': True,
        'device': 'CPU',
    },
    'training_parameters': {
        'max_iterations': 2,
        'gen_iterations': 1,
        'disc_iterations': 1,
        'gen_learning_rate': 0.02,
        'disc_learning_rate': 0.02,
        'print_progress_iterations': 1,
    },
    'data_management': {
        'data_path': '../data/test/',
        'run_id': 'qgan_rh-q3-fake-SPSA-seed0',
        'reset_training_data': True,
        'create_circuits': True,
    },
    'backend_options': {
        'training_precision': 0.03125,
        'data_type': 'double',
        'real_backend_options': {
            'name': 'ibm_basquecountry',
            'channel': 'ibm_quantum_platform',
            'real_estimator_options': {
                'resilience_level': 1,
                'dynamical_decoupling': {'enable': True},
            },
        },
    },
    'embedding_options': {
        'reset_dataset': True,
        'batch_size': 4,
        'randomness': 1.0,
    },
    'eval_options': {
        'method': 'statevector',
        'precision': 0.0,
    },
    'hardware_packing_options': {
        'optimization_level': 1,
        'show_layout': True,
    },
    'gradient_options': {
        'epsilon': 1e-2,
        'batch_size': 4,
    },
}

In [ ]:
#- Load configuration file -#

# Load config file
def load_config_file(filename):
    with open(filename, 'r') as file:
        return yaml.safe_load(file)


if LOAD_CONFIG_FROM_FILE:
    config = load_config_file(configuration_file_path)
    config_path = os.path.dirname(configuration_file_path) + '/'
else:
    config = config_dict

implementation_options = config['implementation_options']
training_parameters = config['training_parameters']
data_management = config['data_management']
backend_options = config['backend_options']
embedding_options = config['embedding_options']
eval_options = config['eval_options']
hardware_packing_options = config['hardware_packing_options']
gradient_options = config['gradient_options']

if not LOAD_CONFIG_FROM_FILE:
    config_path = os.path.join(data_management['data_path'], data_management['run_id']) + '/'
    os.makedirs(config_path, exist_ok=True)

seed = implementation_options['seed']
rng = np.random.default_rng(seed)
np.random.seed(seed)

n_qubits = implementation_options['n_qubits']
random_input = implementation_options['random_input']
execution_type = implementation_options['execution_type']
gradient_method = implementation_options['gradient_method']
device_option = implementation_options['device']

batch_size = int(embedding_options['batch_size'])
embedding_randomness = embedding_options['randomness']
eval_method = eval_options['method']
eval_precision = eval_options['precision']

training_precision = backend_options['training_precision']
data_type = backend_options['data_type']
real_backend_options = backend_options['real_backend_options']
real_backend_name = real_backend_options['name']
real_backend_channel = real_backend_options['channel']
estimator_options = real_backend_options['real_estimator_options']
if 'default_precision' in estimator_options:
    training_precision = estimator_options['default_precision']

if execution_type not in ['real', 'fake']:
    raise ValueError("execution_type must be 'real' or 'fake'.")

print('Configuration loaded')
print('Loaded from file:', LOAD_CONFIG_FROM_FILE)
print('Data path:', config_path)
print('Backend:', real_backend_name)


Configuration loaded
Loaded from file: False
Data path: ../data/test/qgan_rh-q3-fake-SPSA-seed0/
Backend: ibm_basquecountry


In [23]:
#- Create backend -#

# Create Runtime backend and estimator.
if execution_type == 'fake':
    backend = FakeSherbrooke()
else:
    service = QiskitRuntimeService(channel=real_backend_channel)
    backend = service.backend(real_backend_name)
    
session = Session(backend=backend)
estimator = EstimatorV2_rh(mode=session, options=estimator_options)

# Create Aer backend and estimator for noiseless generator evaluation.
eval_backend_options = {
    'method': eval_method,
    'precision': data_type,
    'seed_simulator': seed,
}
eval_backend = AerSimulator(**eval_backend_options)
eval_estimator = EstimatorV2_sim(
    options={
        'default_precision': eval_precision,
        'backend_options': eval_backend.options,
    }
)


# Save backend file
def create_backend_file(backend, filename):
    backend_dict = {
        'timestamp': dt.datetime.now(dt.timezone.utc),
        'configuration': backend.configuration(),
        'properties': backend.properties(),
        'target': backend.target,
        'options': backend.options,
    }
    with open(filename, 'wb') as file:
        pickle.dump(backend_dict, file)

    print('Backend file created.')


create_backend_file(backend, config_path + 'backend.pkl')
print(backend)


Backend file created.


In [24]:
#- Load dataset -#
def apply_curve(x, curve):
    if curve == 'linear':
        return x
    if curve == 'quadratic':
        return x ** 2
    if curve == 'sqrt':
        return np.sqrt(x)
    if curve == 'log':
        return np.log1p(x * 9) / np.log(10)
    if curve == 'exp':
        return (np.exp(x * 3) - 1) / (np.exp(3) - 1)
    if curve == 'sigmoid':
        return 1 / (1 + np.exp(-10 * (x - 0.5)))
    if curve == 'sin':
        return 0.5 * (1 - np.cos(np.pi * x))
    raise ValueError(f'Unknown curve type: {curve}')


def create_gradients(total_pixels, directions=None, curves=None, width=None, height=None):
    if directions is None:
        directions = ['top_left_to_bottom_right']
    if curves is None:
        curves = ['linear', 'quadratic', 'sqrt', 'log', 'exp', 'sigmoid', 'sin']

    if width is None or height is None:
        for h in range(int(np.sqrt(total_pixels)), 0, -1):
            if total_pixels % h == 0:
                width, height = total_pixels // h, h
                break
    elif width * height != total_pixels:
        raise ValueError('Provided width and height do not match total number of pixels.')

    i, j = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
    norm_maps = {
        'left_to_right': np.tile(np.linspace(0, 1, width), (height, 1)),
        'right_to_left': np.tile(np.linspace(1, 0, width), (height, 1)),
        'top_to_bottom': np.tile(np.linspace(0, 1, height)[:, np.newaxis], (1, width)),
        'bottom_to_top': np.tile(np.linspace(1, 0, height)[:, np.newaxis], (1, width)),
        'top_left_to_bottom_right': (i + j) / (width + height - 2),
        'bottom_right_to_top_left': ((height - 1 - i) + (width - 1 - j)) / (width + height - 2),
        'top_right_to_bottom_left': (i + (width - 1 - j)) / (width + height - 2),
        'bottom_left_to_top_right': ((height - 1 - i) + j) / (width + height - 2),
    }

    gradients = []
    for direction in directions:
        if direction not in norm_maps:
            raise ValueError(f'Unknown direction: {direction}')
        for curve in curves:
            gradients.append(apply_curve(norm_maps[direction], curve))
    return np.asarray(gradients).reshape(-1, height, width)


# Create dataset file
def create_dataset_file(filename):
    image_array = create_gradients(n_qubits)
    np.save(filename, image_array)
    print('Dataset file created.')



# Load dataset from file
def load_dataset_file(filename):
    if embedding_options['reset_dataset'] or not os.path.exists(filename):
        create_dataset_file(filename)
    return np.load(filename)


X = load_dataset_file(config_path + 'dataset.npy')
img_h, img_w = X.shape[1:3]

# Show dataset
for idx in range(len(X)):
    plt.subplot(1, len(X), idx + 1)
    plt.imshow(X[idx], cmap='gray')
    plt.axis('off')
plt.show()
print('dataset shape:', X.shape)


Dataset file created.


dataset shape: (7, 1, 7)


In [25]:
#- Angle embedding -#
# Create real data sample circuit
def generate_real_circuit():
    real_weights = ParameterVector('θ_r', n_qubits)
    qc = QuantumCircuit(n_qubits, name='Real')
    for q, parameter in enumerate(real_weights):
        qc.ry(parameter, q)
    return qc


In [26]:
#- Create quantum circuits -#

# Create generator
def generate_generator():
    qc = real_amplitudes(
        n_qubits,
        reps=3,
        parameter_prefix='θ_g',
        name='Generator',
    )
    return qc.decompose()



# Create discriminator
def generate_discriminator():
    qc = efficient_su2(
        n_qubits,
        entanglement='reverse_linear',
        reps=1,
        parameter_prefix='θ_d',
        name='Discriminator',
    ).decompose()

    param_index = qc.num_parameters
    for i in reversed(range(n_qubits - 1)):
        qc.cx(i, n_qubits - 1)
    qc.ry(Parameter(f'θ_d[{param_index}]'), n_qubits - 1)
    param_index += 1
    qc.rz(Parameter(f'θ_d[{param_index}]'), n_qubits - 1)
    return qc



# Create circuits file
def create_circuits_file(filename):
    circuits = [generate_real_circuit(), generate_generator(), generate_discriminator()]
    with open(filename, 'wb') as fd:
        qpy.dump(circuits, fd)
    print('Circuits file created.')



# Load circuits from file
def load_circuits_file(filename):
    if data_management['create_circuits'] or not os.path.exists(filename):
        create_circuits_file(filename)
    with open(filename, 'rb') as fd:
        return qpy.load(fd)


real_circuit, generator_circuit, discriminator_circuit = load_circuits_file(config_path + 'circuits.qpy')
real_params = list(real_circuit.parameters)
gen_params_base = list(generator_circuit.parameters)
disc_params_base = list(discriminator_circuit.parameters)

N_RPARAMS = len(real_params) if random_input else 0
N_GPARAMS = len(gen_params_base)
N_DPARAMS = len(disc_params_base)

print('N_RPARAMS:', N_RPARAMS)
print('N_GPARAMS:', N_GPARAMS)
print('N_DPARAMS:', N_DPARAMS)


Circuits file created.


N_RPARAMS: 7
N_GPARAMS: 28
N_DPARAMS: 30


In [27]:
#- Set up training quantum circuits -#
# Connect base circuits for training
def build_training_templates():
    real_disc = QuantumCircuit(n_qubits, name='real_disc')
    real_disc.compose(real_circuit, inplace=True)
    real_disc.compose(discriminator_circuit, inplace=True)

    ran_gen = QuantumCircuit(n_qubits, name='ran_gen')
    if random_input:
        ran_gen.compose(real_circuit, inplace=True)
    ran_gen.compose(generator_circuit, inplace=True)

    gen_disc = QuantumCircuit(n_qubits, name='gen_disc')
    gen_disc.compose(ran_gen, inplace=True)
    gen_disc.compose(discriminator_circuit, inplace=True)

    return real_disc, gen_disc, ran_gen


real_disc_template, gen_disc_template, gen_eval_template = build_training_templates()

real_disc_template_params = list(real_disc_template.parameters)
gen_disc_template_params = list(gen_disc_template.parameters)
gen_eval_template_params = list(gen_eval_template.parameters)

print('real_disc params:', len(real_disc_template_params))
print('gen_disc params:', len(gen_disc_template_params))
print('gen_eval params:', len(gen_eval_template_params))


real_disc params: 37
gen_disc params: 65
gen_eval params: 35


In [28]:
packed_qubits = batch_size * n_qubits

if batch_size < 1:
    raise ValueError('batch_size must be at least 1.')
if packed_qubits > backend.num_qubits:
    raise ValueError(f'batch_size={batch_size} needs {packed_qubits} qubits, but backend has {backend.num_qubits}.')

print('batch_size:', batch_size)
print('Packed circuit qubits:', packed_qubits)


batch_size: 4
Packed circuit qubits: 28


In [29]:
#- Hardware layout visualization -#
# Get backend layout information.
def get_backend_qubit_positions(backend):
    coords = backend.configuration().coords
    if coords:
        return {idx: (float(x), -float(y)) for idx, (x, y) in enumerate(coords)}

    width = int(np.ceil(np.sqrt(backend.num_qubits)))
    return {idx: (idx % width, -(idx // width)) for idx in range(backend.num_qubits)}


def get_backend_edges():
    coupling_map = backend.target.build_coupling_map()
    edges = {tuple(sorted((int(a), int(b)))) for a, b in coupling_map.get_edges() if a != b}
    return sorted(edges)


def get_isa_two_qubit_edges(circuit):
    edges = set()
    for instruction in circuit.data:
        qubits = [qubit._index for qubit in instruction.qubits]
        if len(qubits) == 2:
            edges.add(tuple(sorted((qubits[0], qubits[1]))))
    return sorted(edges)


# Draw the physical qubits selected by Qiskit's transpiler.
def draw_transpiled_layout(packed_job, title='Transpiled packed layout'):
    positions = get_backend_qubit_positions(backend)
    backend_edges = get_backend_edges()
    isa_edges = get_isa_two_qubit_edges(packed_job['circuit'])
    layout_groups = packed_job['layout_groups']

    colors = plt.get_cmap('tab20')(np.linspace(0, 1, max(len(layout_groups), 1)))
    selected_qubits = set().union(*[set(group) for group in layout_groups])

    fig, ax = plt.subplots(figsize=(14, 7))

    for a, b in backend_edges:
        if a in positions and b in positions:
            ax.plot([positions[a][0], positions[b][0]], [positions[a][1], positions[b][1]], color='0.86', linewidth=0.7, zorder=1)

    idle_qubits = [q for q in positions if q not in selected_qubits]
    ax.scatter(
        [positions[q][0] for q in idle_qubits],
        [positions[q][1] for q in idle_qubits],
        s=28,
        color='0.78',
        edgecolors='white',
        linewidths=0.4,
        zorder=2,
        label='idle',
    )

    for a, b in isa_edges:
        if a in positions and b in positions:
            ax.plot([positions[a][0], positions[b][0]], [positions[a][1], positions[b][1]], color='black', linewidth=2.0, alpha=0.45, zorder=3)

    for copy_index, group in enumerate(layout_groups):
        color = colors[copy_index]
        ax.scatter(
            [positions[q][0] for q in group],
            [positions[q][1] for q in group],
            s=120,
            color=[color],
            edgecolors='black',
            linewidths=0.8,
            zorder=4,
            label=f'copy {copy_index}',
        )
        for local_index, q in enumerate(group):
            ax.text(positions[q][0], positions[q][1], f'{q}\nq{local_index}', ha='center', va='center', fontsize=7, color='black', zorder=5)

    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False)
    plt.tight_layout()
    plt.show()


In [ ]:
#- Packed circuit construction -#
# Classify template parameters by model role
def categorize_template_params(template_params):
    categories = []
    for parameter in template_params:
        if parameter in disc_params_base:
            categories.append(('disc', disc_params_base.index(parameter)))
        elif parameter in gen_params_base:
            categories.append(('gen', gen_params_base.index(parameter)))
        elif parameter in real_params:
            categories.append(('real', real_params.index(parameter)))
        else:
            raise ValueError(f'Unknown parameter in template: {parameter}')
    return categories


# Pack independent logical copies into one circuit.
def pack_parameterized_circuit(template, parameter_prefix):
    template_params = list(template.parameters)
    packed = QuantumCircuit(template.num_qubits * batch_size, name=f'packed_{template.name}')
    logical_params = []
    copy_params = []

    for copy_index in range(batch_size):
        params = ParameterVector(f'{parameter_prefix}_{copy_index}', len(template_params))
        renamed = template.assign_parameters(dict(zip(template_params, params)), inplace=False)
        qargs = list(range(copy_index * template.num_qubits, (copy_index + 1) * template.num_qubits))
        packed.compose(renamed, qubits=qargs, inplace=True)
        logical_params.extend(params)
        copy_params.append(list(params))

    return packed, template_params, categorize_template_params(template_params), logical_params, copy_params


# Build one output observable per packed copy.
def build_output_observables():
    observables = []
    for copy_index in range(batch_size):
        output_qubit = copy_index * n_qubits + (n_qubits - 1)
        observables.append(
            SparsePauliOp.from_sparse_list([('Z', [output_qubit], 1.0)], num_qubits=batch_size * n_qubits)
        )
    return observables


# Recover the physical qubits chosen by Qiskit for each logical copy.
def get_layout_groups(packed_circuit, isa_circuit):
    virtual_to_physical = isa_circuit.layout.initial_layout.get_virtual_bits()
    groups = []

    for copy_index in range(batch_size):
        group = []
        for local_index in range(n_qubits):
            logical_index = copy_index * n_qubits + local_index
            group.append(virtual_to_physical[packed_circuit.qubits[logical_index]])
        groups.append(group)

    return groups


# Let Qiskit choose the physical qubit layout and routing.
def transpile_packed_circuit(packed_circuit, observables):
    start_time = time.time()
    pm = generate_preset_pass_manager(
        optimization_level=hardware_packing_options['optimization_level'],
        backend=backend,
        seed_transpiler=seed,
    )
    print(f'Transpiling {packed_circuit.name}: {packed_circuit.num_qubits} qubits, {packed_circuit.num_parameters} parameters')
    isa_circuit = pm.run(packed_circuit)
    print(f'Transpiled {packed_circuit.name} in {time.time() - start_time:.1f}s')
    isa_observables = [observable.apply_layout(isa_circuit.layout) for observable in observables]
    layout_groups = get_layout_groups(packed_circuit, isa_circuit)
    return isa_circuit, isa_observables, layout_groups


# Prepare circuit, observables and parameter order for Runtime.
def prepare_packed_job(template, parameter_prefix):
    packed_circuit, template_params, categories, logical_params, copy_params = pack_parameterized_circuit(
        template,
        parameter_prefix,
    )
    observables = build_output_observables()
    isa_circuit, isa_observables, layout_groups = transpile_packed_circuit(packed_circuit, observables)

    isa_params = list(isa_circuit.parameters)
    logical_index = {parameter: idx for idx, parameter in enumerate(logical_params)}
    logical_to_isa_indices = [logical_index[parameter] for parameter in isa_params]

    return {
        'template': template,
        'template_params': template_params,
        'categories': categories,
        'circuit': isa_circuit,
        'observables': isa_observables,
        'logical_params': logical_params,
        'copy_params': copy_params,
        'parameters': isa_params,
        'logical_to_isa_indices': logical_to_isa_indices,
        'layout_groups': layout_groups,
    }


packed_real_disc = prepare_packed_job(real_disc_template, 'prd')
packed_gen_disc = prepare_packed_job(gen_disc_template, 'pgd')
print('Packed real_disc parameters:', len(packed_real_disc['parameters']))
print('Packed gen_disc parameters:', len(packed_gen_disc['parameters']))
print('Qiskit layout groups:', packed_real_disc['layout_groups'])

# Physical layout selected by Qiskit after transpilation.
if hardware_packing_options['show_layout']:
    draw_transpiled_layout(packed_real_disc, title='Transpiled packed real_disc layout')
    draw_transpiled_layout(packed_gen_disc, title='Transpiled packed gen_disc layout')


In [ ]:
#- Batch parallelization -#
import torch

# Torch is used only for parameter ownership and optimizer steps. Qiskit Runtime still
# receives NumPy parameter values computed from these tensors.
device = torch.device('cuda' if device_option == 'GPU' and torch.cuda.is_available() else 'cpu')
dtype = torch.float64
torch.manual_seed(seed)


def f_loss(x, label):
    loss = -x * label
    return loss.mean()


def params_to_numpy(values):
    if values is None:
        return None
    if isinstance(values, torch.Tensor):
        return values.detach().cpu().numpy()
    return np.asarray(values, dtype=float)


# Create random input
def generate_random_input(batch_size, num_params):
    if num_params == 0:
        return np.empty((batch_size, 0), dtype=float)
    return 2 * np.pi * rng.random((batch_size, num_params)) * embedding_randomness



# Get real data input
def generate_real_input(batch_size):
    data_indexes = rng.integers(low=0, high=X.shape[0], size=batch_size)
    return X[data_indexes].reshape(batch_size, -1)



# Build parameter values for one packed copy
def build_copy_values(categories, disc_values=None, gen_values=None, real_values=None):
    disc_values = params_to_numpy(disc_values)
    gen_values = params_to_numpy(gen_values)
    real_values = params_to_numpy(real_values)

    values = []
    for category, index in categories:
        if category == 'disc':
            values.append(disc_values[index])
        elif category == 'gen':
            values.append(gen_values[index])
        elif category == 'real':
            values.append(real_values[index])
        else:
            raise ValueError(f'Unknown category: {category}')
    return np.asarray(values, dtype=float)



# Reorder packed parameters for the transpiled ISA circuit
def build_packed_parameter_row(packed_job, per_copy_values):
    per_copy_values = np.asarray(per_copy_values, dtype=float)
    if per_copy_values.shape[0] != batch_size:
        raise ValueError(f'Expected {batch_size} copies, got {per_copy_values.shape[0]}.')
    logical_row = per_copy_values.reshape(1, -1)
    return logical_row[:, packed_job['logical_to_isa_indices']]



# Run Runtime estimator values
def run_estimator_values(packed_job, per_copy_values, observables=None):
    parameter_values = build_packed_parameter_row(packed_job, per_copy_values)
    observables = packed_job['observables'] if observables is None else observables
    pub = (packed_job['circuit'], observables, parameter_values)
    job = estimator.run([pub], precision=training_precision)
    return np.asarray(job.result()[0].data.evs, dtype=float)



# Select trainable parameters for one packed copy
def selected_copy_parameters(packed_job, category_name, copy_index):
    selected = []
    for param, (category, _) in zip(packed_job['copy_params'][copy_index], packed_job['categories']):
        if category == category_name:
            selected.append(param)
    return selected



# Run gradient jobs for all packed copies
def run_gradient_values(gradient, packed_job, per_copy_values, category_name):
    parameter_values = build_packed_parameter_row(packed_job, per_copy_values)[0]
    circuits = []
    observables = []
    parameter_rows = []
    parameter_sets = []

    for copy_index in range(batch_size):
        circuits.append(packed_job['circuit'])
        observables.append(packed_job['observables'][copy_index])
        parameter_rows.append(parameter_values)
        parameter_sets.append(selected_copy_parameters(packed_job, category_name, copy_index))

    result = gradient.run(circuits, observables, parameter_rows, parameters=parameter_sets).result()
    return [np.asarray(grad, dtype=float) for grad in result.gradients]


if gradient_method == 'SPSA':
    gradient = SPSAEstimatorGradient(
        estimator=estimator,
        epsilon=float(gradient_options['epsilon']),
        batch_size=int(gradient_options['batch_size']),
        seed=seed,
        precision=training_precision,
    )
elif gradient_method == 'PSR':
    gradient = ParamShiftEstimatorGradient(estimator=estimator, precision=training_precision)
else:
    raise ValueError("Real-hardware Runtime notebook supports gradient_method 'SPSA' or 'PSR'.")


# Aer generator evaluation helpers
eval_generator_categories = categorize_template_params(gen_eval_template_params)
eval_generator_observables = [
    SparsePauliOp.from_sparse_list([('Z', [local_q], 1.0)], num_qubits=n_qubits)
    for local_q in range(n_qubits)
]



# Generate input for generator evaluation
def generator_eval_rows(batch_size, random_inputs=None):
    if random_inputs is None:
        random_inputs = generate_random_input(batch_size, N_RPARAMS)

    rows = []
    for copy_index in range(batch_size):
        real_values = random_inputs[copy_index] if N_RPARAMS else None
        rows.append(
            build_copy_values(
                eval_generator_categories,
                gen_values=gen_params,
                real_values=real_values,
            )
        )
    return np.asarray(rows, dtype=float)



# Evaluate generator locally with Aer
def run_aer_generator_eval(per_copy_values):
    parameter_values = np.asarray(per_copy_values, dtype=float)[:, np.newaxis, :]
    pub = (gen_eval_template, eval_generator_observables, parameter_values)
    job = eval_estimator.run([pub], precision=eval_precision)
    return np.asarray(job.result()[0].data.evs, dtype=float)

In [ ]:
#- Restore parameters and optimizer states -#
gen_learning_rate = training_parameters['gen_learning_rate']
disc_learning_rate = training_parameters['disc_learning_rate']


# Reset all data training
def create_training_data_file(filename):
    init_gen_params = rng.uniform(low=-np.pi, high=np.pi, size=N_GPARAMS) * 0.1
    init_disc_params = rng.uniform(low=-np.pi, high=np.pi, size=N_DPARAMS) * 0.1

    gen_param = torch.nn.Parameter(torch.tensor(init_gen_params, dtype=dtype, device=device))
    disc_param = torch.nn.Parameter(torch.tensor(init_disc_params, dtype=dtype, device=device))
    optimizer_g = torch.optim.Adam([gen_param], lr=gen_learning_rate)
    optimizer_d = torch.optim.Adam([disc_param], lr=disc_learning_rate)

    data = {
        'init_gen_params': init_gen_params,
        'init_disc_params': init_disc_params,
        'gen_params': gen_param.detach().cpu(),
        'disc_params': disc_param.detach().cpu(),
        'best_gen_params': init_gen_params.copy(),
        'current_epoch': 0,
        'metrics': {'gloss': {}, 'dloss': {}, 'eval': {}, 'times': {}},
        'rng_state': rng.bit_generator.state,
        'np_random_state': np.random.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'optimizer_g_state': optimizer_g.state_dict(),
        'optimizer_d_state': optimizer_d.state_dict(),
    }
    torch.save(data, filename)
    print('Training data file created.')



def load_legacy_training_data_file(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


# Load parameters and training states from file
def load_training_data_file(filename):
    legacy_filename = config_path + 'rh_packed_training_data.pkl'
    if data_management['reset_training_data'] or not os.path.exists(filename):
        if not data_management['reset_training_data'] and os.path.exists(legacy_filename):
            return load_legacy_training_data_file(legacy_filename)
        create_training_data_file(filename)
    return torch.load(filename, weights_only=False, map_location=device)


training_data_path = config_path + 'training_data.pth'
params = load_training_data_file(training_data_path)

if 'rng_state' in params:
    rng.bit_generator.state = params['rng_state']
if 'np_random_state' in params:
    np.random.set_state(params['np_random_state'])
if 'torch_rng_state' in params:
    torch.set_rng_state(params['torch_rng_state'])


gen_params = torch.nn.Parameter(torch.as_tensor(params['gen_params'], dtype=dtype, device=device).clone().detach())
disc_params = torch.nn.Parameter(torch.as_tensor(params['disc_params'], dtype=dtype, device=device).clone().detach())
best_gen_params = params_to_numpy(params['best_gen_params']).copy()
current_epoch = int(params.get('current_epoch', 0))
epoch = current_epoch - 1
metrics = params.get('metrics', {'gloss': {}, 'dloss': {}, 'eval': {}, 'times': {}})
gloss = metrics['gloss']
gen_loss = list(gloss.values())[-1] if gloss else None
dloss = metrics['dloss']
disc_loss = list(dloss.values())[-1] if dloss else None
eval_history = metrics['eval']
times = metrics['times']
min_eval = min(eval_history.values()) if eval_history else float('inf')

optimizer_g = torch.optim.Adam([gen_params], lr=gen_learning_rate)
optimizer_d = torch.optim.Adam([disc_params], lr=disc_learning_rate)
if 'optimizer_g_state' in params:
    optimizer_g.load_state_dict(params['optimizer_g_state'])
if 'optimizer_d_state' in params:
    optimizer_d.load_state_dict(params['optimizer_d_state'])

print('Starting epoch:', current_epoch)
print('Generator params:', tuple(gen_params.shape), gen_params.device, gen_params.dtype)
print('Discriminator params:', tuple(disc_params.shape), disc_params.device, disc_params.dtype)

In [ ]:
#- Manage training interruption -#
# Class to manage training interruption
class Interrupter:
    def __init__(self):
        self.kill_now = False
        signal.signal(signal.SIGINT, self.handle_signal)
        signal.signal(signal.SIGTERM, self.handle_signal)

    def handle_signal(self, signum, frame):
        print('Interrupter: Graceful exit requested.')
        self.kill_now = True


interrupter = Interrupter()


In [ ]:
#- Evualuation method -#

# Evaluate specific gradient (top-left to bottom-right) for small images
def evaluate(gen_dists, targets=None):
    batch = gen_dists.reshape(-1, img_h, img_w)

    h_diff = batch[:, :, 1:] - batch[:, :, :-1]
    v_diff = batch[:, 1:, :] - batch[:, :-1, :]

    h_penalty = (-h_diff.clamp(max=0)).mean()
    v_penalty = (-v_diff.clamp(max=0)).mean()

    penalty = 0.5 * (h_penalty + v_penalty)
    return penalty.item()


# Evaluate batch of generated samples
def batch_evaluation(batch_size):
    random_inputs = generate_random_input(batch_size, N_RPARAMS)
    rows = generator_eval_rows(batch_size, random_inputs)
    evs = run_aer_generator_eval(rows).reshape(batch_size, n_qubits)
    fake_outputs = torch.as_tensor(evs, dtype=dtype, device=device)
    return evaluate(fake_outputs)

In [ ]:
#- Forward and backward pass -#
# Generate discriminator inputs
def discriminator_rows(real_inputs, random_inputs=None):
    real_rows = []
    fake_rows = []
    for copy_index in range(batch_size):
        real_rows.append(
            build_copy_values(
                packed_real_disc['categories'],
                disc_values=disc_params,
                real_values=real_inputs[copy_index],
            )
        )
        fake_real_values = random_inputs[copy_index] if N_RPARAMS else None
        fake_rows.append(
            build_copy_values(
                packed_gen_disc['categories'],
                disc_values=disc_params,
                gen_values=gen_params,
                real_values=fake_real_values,
            )
        )
    return np.asarray(real_rows), np.asarray(fake_rows)



# Generate input for generator
def generator_rows(random_inputs=None):
    rows = []
    for copy_index in range(batch_size):
        real_values = random_inputs[copy_index] if N_RPARAMS else None
        rows.append(
            build_copy_values(
                packed_gen_disc['categories'],
                disc_values=disc_params,
                gen_values=gen_params,
                real_values=real_values,
            )
        )
    return np.asarray(rows)



# Average gradients across packed copies
def average_gradients(copy_gradients, width):
    if not copy_gradients:
        return np.zeros(width, dtype=float)
    return np.mean(np.asarray(copy_gradients, dtype=float), axis=0)



# Discriminator pass
def disc_pass():
    optimizer_d.zero_grad()

    real_inputs = generate_real_input(batch_size)
    random_inputs = generate_random_input(batch_size, N_RPARAMS)
    real_rows, fake_rows = discriminator_rows(real_inputs, random_inputs)

    real_output = run_estimator_values(packed_real_disc, real_rows).reshape(-1)
    fake_output = run_estimator_values(packed_gen_disc, fake_rows).reshape(-1)

    real_grads = run_gradient_values(gradient, packed_real_disc, real_rows, 'disc')
    fake_grads = run_gradient_values(gradient, packed_gen_disc, fake_rows, 'disc')

    grad_dcost_adjoint = -average_gradients(real_grads, N_DPARAMS) + average_gradients(fake_grads, N_DPARAMS)
    disc_params.grad = torch.tensor(grad_dcost_adjoint, dtype=torch.float64, device=device)
    optimizer_d.step()

    real_output = torch.as_tensor(real_output, dtype=dtype, device=device)
    fake_output = torch.as_tensor(fake_output, dtype=dtype, device=device)
    real_loss = f_loss(real_output, torch.ones_like(real_output))
    fake_loss = f_loss(fake_output, -torch.ones_like(fake_output))
    disc_loss = (real_loss.item() + fake_loss.item() - 2) / 4
    return float(disc_loss)



# Generator pass
def gen_pass():
    optimizer_g.zero_grad()

    random_inputs = generate_random_input(batch_size, N_RPARAMS)
    rows = generator_rows(random_inputs)
    fake_output = run_estimator_values(packed_gen_disc, rows).reshape(-1)
    gen_grads = run_gradient_values(gradient, packed_gen_disc, rows, 'gen')

    grad_gcost_adjoint = -average_gradients(gen_grads, N_GPARAMS)
    gen_params.grad = torch.tensor(grad_gcost_adjoint, dtype=torch.float64, device=device)
    optimizer_g.step()

    fake_output = torch.as_tensor(fake_output, dtype=dtype, device=device)
    gen_loss = f_loss(fake_output, torch.ones_like(fake_output))
    gen_loss = (gen_loss.item() - 1) / 2
    return gen_loss


# Copy parameters
def copy_params():
    return gen_params.detach().cpu().numpy().copy()

In [ ]:
#- Training -#
D_STEPS = training_parameters['disc_iterations']
G_STEPS = training_parameters['gen_iterations']
MAX_ITER = training_parameters['max_iterations']
PRINT_EVERY = training_parameters['print_progress_iterations']

if PRINT_EVERY:
    TABLE_HEADERS = 'Epoch | Generator cost | Discriminator cost | Eval | Best eval | Time |'
    print(TABLE_HEADERS)

start_time = time.time()
prev_times = 0.0
#--- Training loop ---#
try: # In case of interruption
    for epoch in range(current_epoch, MAX_ITER + 1):
        #--- Quantum discriminator parameter updates ---#
        for _ in range(D_STEPS):
            disc_loss = disc_pass()
            dloss[epoch] = disc_loss


        #--- Quantum generator parameter updates ---#
        for _ in range(G_STEPS):
            gen_loss = gen_pass()
            gloss[epoch] = gen_loss


        #--- Track Eval and save best performing generator weights ---#
        current_eval = batch_evaluation(batch_size * 2)
        eval_history[epoch] = current_eval
        if current_eval < min_eval:
            min_eval = current_eval
            best_gen_params = copy_params()


        # Calculate time
        cur_time = time.time() - start_time
        times[epoch] = cur_time
        start_time = time.time()


        #--- Print progress ---#
        if PRINT_EVERY and epoch % PRINT_EVERY == 0:
            now_times = sum(times.values())
            for header, val in zip(
                TABLE_HEADERS.split('|'),
                (epoch, gen_loss, disc_loss, current_eval, min_eval, now_times - prev_times),
            ):
                print(f'{val:.3g} '.rjust(len(header)), end='|')
            print()
            prev_times = now_times


        # In case of interruption
        if interrupter.kill_now:
            print('Interrupter: Graceful exit triggered. Breaking loop.')
            break

#--- Save parameters and optimizer states data ---#
finally:
    params = {
        'init_gen_params': params['init_gen_params'],
        'init_disc_params': params['init_disc_params'],
        'best_gen_params': best_gen_params,
        'gen_params': gen_params.detach().cpu(),
        'disc_params': disc_params.detach().cpu(),
        'current_epoch': epoch + 1,
        'metrics': {
            'gloss': gloss,
            'dloss': dloss,
            'eval': eval_history,
            'times': times,
        },
        'rng_state': rng.bit_generator.state,
        'np_random_state': np.random.get_state(),
        'torch_rng_state': torch.get_rng_state(),
        'optimizer_g_state': optimizer_g.state_dict(),
        'optimizer_d_state': optimizer_d.state_dict(),
        'layout_groups': {
            'real_disc': packed_real_disc['layout_groups'],
            'gen_disc': packed_gen_disc['layout_groups'],
        },
        'batch_size': batch_size,
    }
    torch.save(params, training_data_path)
    if session is not None:
        session.close()

    eval_data = list(eval_history.values()) if eval_history else [0]
    print(
        'Training complete:',
        '\n   Data path:', config_path,
        '\n   Best eval:', np.min(eval_data),
        'in epoch', int(np.argmin(eval_data)),
        '\n   Total time:', sum(times.values()),
    )

In [ ]:
#- Results -#
plt.figure(figsize=(10, 3))
plt.subplot(1, 3, 1)
plt.plot(list(gloss.keys()), list(gloss.values()))
plt.title('Generator')
plt.subplot(1, 3, 2)
plt.plot(list(dloss.keys()), list(dloss.values()))
plt.title('Discriminator')
plt.subplot(1, 3, 3)
plt.plot(list(eval_history.keys()), list(eval_history.values()))
plt.title('Eval')
plt.tight_layout()
plt.show()

print('Final generator params:', gen_params.detach().cpu().numpy())
print('Best generator params:', best_gen_params)
print('Final discriminator params:', disc_params.detach().cpu().numpy())